# KL Divergence Wstimates

**Core Problem:** estimate the gap between the current policy p and a reference policy q used as the KL penalty in PPO/GRPO
- q usualy a frozen SFT Model


In [9]:
import torch

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = "LiquidAI/LFM2.5-230M"
model = AutoModelForCausalLM.from_pretrained(model_id,device_map="mps")
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights:   0%|          | 0/132 [00:00<?, ?it/s]

In [17]:
prompt = "Hello "
input = tokenizer(prompt,return_tensors="pt").to(model.device)
logits = model(**input).logits
with torch.no_grad():
    log_probas = torch.softmax(logits,dim=-1)
print(log_probas)

tensor([[[5.2751e-10, 9.6484e-01, 8.3618e-03,  ..., 4.9477e-10,
          4.9477e-10, 4.9477e-10],
         [3.4925e-10, 1.5832e-08, 1.9670e-05,  ..., 3.6016e-10,
          3.6016e-10, 3.6016e-10],
         [6.9558e-09, 1.3784e-06, 2.8849e-05,  ..., 7.1886e-09,
          7.1886e-09, 7.1886e-09]]], device='mps:0', dtype=torch.bfloat16)


In [23]:
def kl_penalty(policy_log_probs,ref_log_probs):
    return (policy_log_probs - ref_log_probs).mean()

In [25]:
policy_log_probs = torch.rand_like(log_probas)
ref_log_probs = log_probas
kl_penalty(policy_log_probs, ref_log_probs).item()

0.5